In [ ]:
!pip install huggingface_hub
!pip install datasets
!pip install transformers

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import torch
from transformers import AutoModel, BertTokenizerFast
from torch.optim import AdamW
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset_train = load_dataset("toxigen/toxigen-data", split = "train")
dataset_test = load_dataset("toxigen/toxigen-data", split = "test")

dataset_train, dataset_test

In [ ]:
bert = AutoModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

Pre-procesamos los textos para no tener que hacerlo durante el training. Los textos se tokenizan y se hace padding de ceros para que todos los batches tengan el mismo tamaño. Los labels se normalizan entre 0 y 1.

In [ ]:
train_text = dataset_train[:8000]["text"]
train_labels = torch.tensor(dataset_train[:8000]["toxicity_human"]) / 5.0
train_tokens = tokenizer.batch_encode_plus(
    train_text,
    max_length=30,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

val_text = dataset_train[8000:]["text"]
val_labels = torch.tensor(dataset_train[8000:]["toxicity_human"]) / 5.0
val_tokens = tokenizer.batch_encode_plus(
    val_text,
    max_length=30,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

test_text   = list(dataset_test["text"])
test_labels = torch.tensor(dataset_test["toxicity_human"]) / 5.0

test_tokens = tokenizer(
    test_text,
    max_length=30,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

In [ ]:
print(train_tokens["input_ids"][0])

Nos interesa quedarnos con los ids de los tokens y las máscaras de atención que van a indicar que porción corresponde a tokens y cual corresponde a padding.

In [ ]:
train_seq = torch.tensor(train_tokens['input_ids'])
train_mask = torch.tensor(train_tokens['attention_mask'])

val_seq = torch.tensor(val_tokens['input_ids'])
val_mask = torch.tensor(val_tokens['attention_mask'])

test_seq = torch.tensor(test_tokens['input_ids'])
test_mask = torch.tensor(test_tokens['attention_mask'])

In [ ]:
batch_size = 1000

train_data = TensorDataset(train_seq, train_mask, train_labels)
train_dataloader = DataLoader(train_data, batch_size = batch_size)

val_data = TensorDataset(val_seq, val_mask, val_labels)
val_dataloader = DataLoader(val_data, batch_size = batch_size)

Vamos a freezar todos los parámetros de Bert. Solo nos interesa finetunear el final de la arquitectura

In [ ]:
for param in bert.parameters():
    param.requires_grad = False

**Definir capas extras para la regresión. Se espera que la salida sea un número entre 0 y 1 que indica la toxicidad del texto. Completar arquitectura y método forward.**

In [ ]:
class BERT_toxic(nn.Module):

    def __init__(self, bert):

        super(BERT_toxic, self).__init__()

        self.bert = bert

        ...

    def forward(self, sent_id, mask):

        _, cls_hs = self.bert(sent_id, attention_mask = mask, return_dict = False)    # salida del modelo bert : (batch, 768)

        ...

        return x

In [ ]:
model = BERT_toxic(bert)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr = 1e-4)
criterion = nn.MSELoss()

Podemos ver la cantidad de parámetros del modelo freezado

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model)

In [ ]:
epochs = 1000
losses = []

model.train()

for epoch in range(epochs):

    total_loss = 0

    for step, batch in enumerate(train_dataloader):

        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        model.zero_grad()

        preds = model(sent_id, mask).squeeze(1)

        loss = criterion(preds, labels)
        total_loss = total_loss + loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_dataloader)
    losses.append(avg_loss)

    if epoch % 10 == 0: print(f"epoch: {epoch} | {loss.item():.4f}")

In [ ]:
plt.plot(losses)
plt.xlabel("epochs")
plt.ylabel("avg loss")

In [ ]:
with torch.no_grad():

    preds = model(test_seq.to(device), test_mask.to(device))
    preds = preds.detach().cpu().squeeze(1)

    accuracy = criterion(preds, test_labels).item()

    print(f"Test MSE : {accuracy:.4f}")

test_text[:10]

In [ ]:
preds[:10]